# Projet SM604 - De la classification des chiffres manuscrits à la détection de cancers du sein
## Mathématiques pour le Machine Learning - EFREI Paris 2025-2026
### Sabrina El Hassani, Aude Labat, Thomas Duriaud, Paul Fontaine, Evan Ladeira

---

## Partie 3 - Application au diagnostic médical (CBIS-DDSM)

Ce notebook est organisé en quatre sections :

**Section 3.1** Chargement et prétraitement du dataset CBIS-DDSM (Kaggle)

**Section 3.2** Architectures denses (linéaire, H=1, H=2) adaptées à la classification binaire

**Section 3.3** CNN adapté au diagnostic médical (Option B : PyTorch)

**Section 3.4** Analyse de la matrice de confusion et discussion médicale

Les fonctions communes (softmax, cross_entropy, one_hot, train_lin, train_h1, train_h2) sont importées depuis `utils.py`.

---

**Structure du dataset Kaggle (`awsaf49/cbis-ddsm-breast-cancer-image-dataset`) :**
```
cbis-ddsm-breast-cancer-image-dataset/
├── csv/
│   ├── mass_case_description_train_set.csv   ← étiquettes + chemins train
│   ├── mass_case_description_test_set.csv    ← étiquettes + chemins test
│   ├── calc_case_description_train_set.csv   ← (non utilisé ici)
│   └── dicom_info.csv                        ← (non utilisé ici)
└── jpeg/
    └── 1.3.6.1.4.1.9590.100.1.2.XXX/        ← dossiers nommés par UID DICOM
        └── 1-211.jpg                         ← images JPEG
```

**Objectif :** classification binaire **bénin (0) vs malin (1)** sur les masses mammaires.
- `BENIGN` + `BENIGN_WITHOUT_CALLBACK` → **0**
- `MALIGNANT` → **1**

---
## Section 3.1 - Chargement et prétraitement de CBIS-DDSM

### Particularité de ce dataset

Contrairement à MNIST et CIFAR-10, les chemins dans le CSV sont des chemins **absolus** issus de Google Drive :
```
/content/drive/MyDrive/Data/cbis-ddsm.../jpeg/1.3.6.1.4.1.9590.100.1.2.XXX/1-211.jpg
```
On en extrait uniquement la partie relative `jpeg/UID/fichier.jpg` pour construire le chemin local.

**Trois défis spécifiques :**

**1. Matching CSV ↔ images** via extraction du chemin relatif depuis la colonne `image file path`.

**2. Déséquilibre des classes** : il y a naturellement plus de cas bénins que malins.

**3. Redimensionnement** : les images JPEG font plusieurs centaines de pixels, on les réduit à 128×128.

### Imports

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from PIL import Image
from sklearn.metrics import confusion_matrix

from utils import *   # softmax, cross_entropy, one_hot, train_lin, train_h1, train_h2

np.random.seed(42)
torch.manual_seed(42)
print("Imports OK")

### Chemins du dataset

Adapter `DATASET_ROOT` au dossier où vous avez extrait le zip Kaggle.

In [ ]:
# --- Adapter ce chemin au dossier extrait depuis Kaggle ---
DATASET_ROOT = "cbis-ddsm-breast-cancer-image-dataset"   # dossier racine du zip Kaggle
CSV_DIR      = os.path.join(DATASET_ROOT, "csv")
JPEG_DIR     = os.path.join(DATASET_ROOT, "jpeg")
IMG_SIZE     = 128   # redimensionnement cible (128x128)

# Vérification que les dossiers existent
for d in [CSV_DIR, JPEG_DIR]:
    existe = os.path.isdir(d)
    print(f"  {d} : {'OK' if existe else 'INTROUVABLE — vérifier DATASET_ROOT'}")

### Chargement du CSV et extraction des chemins relatifs

La colonne `image file path` contient un chemin absolu de la forme :
```
/content/drive/.../jpeg/1.3.6.1.4.1.9590.100.1.2.XXX/1-211.jpg
```
On extrait la sous-chaîne à partir de `jpeg/` pour obtenir le chemin relatif local.

In [ ]:
def extraire_chemin_relatif(chemin_csv):
    """
    Extrait la partie relative du chemin depuis la colonne 'image file path' du CSV.
    Le CSV contient des chemins absolus issus de Google Drive, on garde uniquement
    ce qui commence à 'jpeg/' pour construire le chemin local.

    Exemple :
      Entrée  : '/content/drive/MyDrive/Data/.../jpeg/1.3.6.../1-211.jpg'
      Sortie  : 'jpeg/1.3.6.../1-211.jpg'

    chemin_csv : str
    Retourne   : str chemin relatif à partir de 'jpeg/'
    """
    chemin_nettoye = chemin_csv.strip().replace("\\", "/")
    # On cherche l'occurrence de 'jpeg/' dans le chemin
    idx = chemin_nettoye.find("jpeg/")
    if idx != -1:
        return chemin_nettoye[idx:]   # 'jpeg/UID/fichier.jpg'
    return chemin_nettoye             # fallback si 'jpeg/' absent


def binariser_pathologie(pathologie):
    """
    Convertit la colonne pathology en étiquette binaire.
    BENIGN et BENIGN_WITHOUT_CALLBACK → 0 (bénin)
    MALIGNANT                          → 1 (malin)

    pathologie : str
    Retourne   : int (0 ou 1), -1 si valeur inconnue
    """
    p = str(pathologie).strip().upper()
    if p in ["BENIGN", "BENIGN_WITHOUT_CALLBACK"]:
        return 0
    elif p == "MALIGNANT":
        return 1
    return -1   # valeur sentinelle


def charger_csv(csv_path):
    """
    Charge un CSV CBIS-DDSM et retourne un DataFrame avec :
    - 'chemin_local' : chemin relatif vers l'image JPEG (à partir de jpeg/)
    - 'label'        : 0 (bénin) ou 1 (malin)

    csv_path : str
    Retourne : pd.DataFrame filtré (sans lignes à label inconnu)
    """
    df = pd.read_csv(csv_path)

    # Extraction du chemin relatif
    df["chemin_local"] = df["image file path"].apply(extraire_chemin_relatif)

    # Binarisation de la pathologie
    df["label"] = df["pathology"].apply(binariser_pathologie)

    # Suppression des cas avec étiquette inconnue
    df = df[df["label"] != -1].reset_index(drop=True)

    return df


# Chargement des deux CSV
df_train = charger_csv(os.path.join(CSV_DIR, "mass_case_description_train_set.csv"))
df_test  = charger_csv(os.path.join(CSV_DIR, "mass_case_description_test_set.csv"))

print(f"CSV train : {len(df_train)} lignes")
print(f"CSV test  : {len(df_test)} lignes")
print(f"\nDistribution train : Bénin={int((df_train['label']==0).sum())}  Malin={int((df_train['label']==1).sum())}")
print(f"Distribution test  : Bénin={int((df_test['label']==0).sum())}   Malin={int((df_test['label']==1).sum())}")
print(f"\nExemple chemin extrait : {df_train['chemin_local'].iloc[0]}")

### Chargement et redimensionnement des images

Les images JPEG du dataset sont en niveaux de gris (1 canal).  
On les charge avec PIL, on les redimensionne à 128×128 et on normalise entre 0 et 1 en divisant par 255.

**Note :** le CSV contient parfois plusieurs lignes pour la même image (cas avec plusieurs anomalies).  
On déduplique sur `chemin_local` pour ne charger chaque image qu'une seule fois.

In [ ]:
def charger_image(chemin_local, jpeg_dir, taille=128):
    """
    Charge une image JPEG CBIS-DDSM, la convertit en niveaux de gris,
    la redimensionne à (taille x taille) et normalise entre 0 et 1.

    chemin_local : str, chemin relatif depuis jpeg_dir (ex. 'jpeg/UID/1-211.jpg')
    jpeg_dir     : str, chemin vers le dossier racine contenant 'jpeg/'
    taille       : int, côté du carré cible
    Retourne     : array (taille, taille) float32, ou None si fichier absent
    """
    chemin_complet = os.path.join(jpeg_dir, chemin_local)

    # Certains chemins commencent par 'jpeg/' et jpeg_dir pointe déjà vers le dossier parent
    # On essaie les deux variantes pour robustesse
    if not os.path.exists(chemin_complet):
        # Variante : chemin_local commence peut-être par 'jpeg/', on enlève ce préfixe
        chemin_alt = os.path.join(jpeg_dir, chemin_local.lstrip("jpeg/"))
        if os.path.exists(chemin_alt):
            chemin_complet = chemin_alt
        else:
            return None   # fichier introuvable

    try:
        img = Image.open(chemin_complet).convert("L")          # L = niveaux de gris
        img = img.resize((taille, taille), Image.BICUBIC)      # redimensionnement bicubique
        return np.array(img, dtype=np.float32) / 255.0         # normalisation [0, 1]
    except Exception:
        return None


def charger_dataset(df, jpeg_dir, taille=128):
    """
    Charge toutes les images d'un DataFrame CSV CBIS-DDSM.
    Déduplique d'abord sur 'chemin_local' (une image peut avoir plusieurs anomalies).

    df       : pd.DataFrame avec colonnes 'chemin_local' et 'label'
    jpeg_dir : str, dossier racine contenant les images
    taille   : int, côté du carré cible
    Retourne : X array (n, taille, taille), y array (n,)
    """
    # Déduplication : en cas d'anomalies multiples, on garde la première ligne
    # (même image, même étiquette, pas besoin de la charger deux fois)
    df_unique = df.drop_duplicates(subset="chemin_local").reset_index(drop=True)
    print(f"  Après déduplication : {len(df_unique)} images uniques sur {len(df)} lignes CSV")

    images, labels = [], []
    n_manquantes = 0

    for i, row in df_unique.iterrows():
        img = charger_image(row["chemin_local"], jpeg_dir, taille)
        if img is not None:
            images.append(img)
            labels.append(row["label"])
        else:
            n_manquantes += 1

        if (i + 1) % 100 == 0:
            print(f"    {len(images)} chargées...", end="\r")

    print(f"    {len(images)} images chargées, {n_manquantes} introuvables.")
    return np.array(images, dtype=np.float32), np.array(labels, dtype=int)


print("Chargement du train...")
X_raw_train, y_train = charger_dataset(df_train, DATASET_ROOT, taille=IMG_SIZE)

print("\nChargement du test...")
X_raw_test,  y_test  = charger_dataset(df_test,  DATASET_ROOT, taille=IMG_SIZE)

print(f"\nTrain : {X_raw_train.shape}  |  Test : {X_raw_test.shape}")
print(f"Pixels : min={X_raw_train.min():.3f}  max={X_raw_train.max():.3f}")
print(f"\nDistribution train : Bénin={int((y_train==0).sum())}  Malin={int((y_train==1).sum())}")
print(f"Distribution test  : Bénin={int((y_test==0).sum())}   Malin={int((y_test==1).sum())}")

### Visualisation des mammographies

On affiche quelques images de chaque classe pour vérifier le chargement et observer la complexité du dataset.  
Contrairement à MNIST (chiffres nets) ou CIFAR-10 (objets colorés en 32×32),  
les mammographies présentent des textures subtiles et une très grande variabilité d'aspect entre patients.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for classe, titre, couleur in [(0, "Bénin",  "green"), (1, "Malin", "red")]:
    indices_classe = np.where(y_train == classe)[0]
    echantillon    = np.random.choice(indices_classe, size=5, replace=False)
    for j, idx in enumerate(echantillon):
        ax = axes[classe][j]
        ax.imshow(X_raw_train[idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(titre, fontsize=10, color=couleur)
        ax.axis("off")

plt.suptitle("Échantillon CBIS-DDSM - Masses mammaires 128×128", fontsize=13)
plt.tight_layout()
plt.savefig("cbis_echantillon.png", dpi=100, bbox_inches="tight")
plt.show()

### Déséquilibre des classes et poids

En diagnostic médical, le déséquilibre des classes a des conséquences directes sur les métriques.  
Un modèle qui prédirait toujours "bénin" obtiendrait un faible taux d'erreur global mais serait médicalement inutile.

On calcule les poids inverses pour pénaliser davantage les erreurs sur les cas malins :
$$w_c = \frac{n_{\text{total}}}{2 \cdot n_c}$$

Ainsi une erreur sur un cas malin coûte $w_1 / w_0$ fois plus qu'une erreur sur un cas bénin.

In [ ]:
n_total_tr = len(y_train)
n_benin_tr = int((y_train == 0).sum())
n_malin_tr = int((y_train == 1).sum())

w_benin = n_total_tr / (2 * n_benin_tr)   # poids classe 0 (bénin)
w_malin = n_total_tr / (2 * n_malin_tr)   # poids classe 1 (malin)

print(f"Distribution train : Bénin={n_benin_tr}  Malin={n_malin_tr}  Total={n_total_tr}")
print(f"Poids de classe    : w_bénin={w_benin:.3f}  w_malin={w_malin:.3f}")
print(f"Rapport            : une erreur sur un malin est pénalisée {w_malin/w_benin:.1f}x plus.")

# Tenseur de poids pour PyTorch (Section 3.3)
class_weights_torch = torch.tensor([w_benin, w_malin], dtype=torch.float32)

### Aplatissement pour les architectures denses

Pour les modèles linéaire et à couches cachées (Section 3.2), on aplatit chaque image 128×128  
en vecteur $\vec{x} \in \mathbb{R}^{16384}$ (car $128 \times 128 = 16\,384$).

Les données sont déjà normalisées entre 0 et 1 lors du chargement.  
Pour le CNN (Section 3.3), on conservera la forme 2D.

In [ ]:
N_PIXELS = IMG_SIZE * IMG_SIZE   # 16 384 pour 128x128

# Aplatissement : (n, 128, 128) → (n, 16384)
X_train_flat = X_raw_train.reshape(len(X_raw_train), -1)   # (n_train, 16384)
X_test_flat  = X_raw_test.reshape(len(X_raw_test),  -1)    # (n_test,  16384)

print(f"X_train_flat : {X_train_flat.shape}")
print(f"X_test_flat  : {X_test_flat.shape}")

# One-hot pour la classification binaire C=2
Y_train_oh = one_hot(y_train, C=2)   # (n_train, 2)
Y_test_oh  = one_hot(y_test,  C=2)   # (n_test,  2)

print(f"\nY_train_oh : {Y_train_oh.shape}")
print(f"Exemple bénin (0) : {Y_train_oh[y_train==0][0]}")
print(f"Exemple malin (1) : {Y_train_oh[y_train==1][0]}")

---
## Section 3.2 - Architectures denses adaptées à la classification binaire

On réutilise les fonctions de `utils.py` construites pour MNIST et CIFAR-10.  
La seule adaptation est le **nombre de classes** : $C = 2$ au lieu de $C = 10$.

Les architectures sont :
- **Linéaire** : $\vec{o} = A\vec{x} + b$ avec $A \in \mathbb{R}^{2 \times 16384}$
- **H=1** : couche cachée de 128 neurones ReLU, puis couche de sortie à 2 neurones
- **H=2** : deux couches cachées (128 puis 64 neurones) avec ReLU

La cross-entropy et le softmax s'appliquent identiquement avec $C=2$.

**Note :** le learning rate est réduit à 0.01 (vs 0.1 sur MNIST) car le dataset est beaucoup plus petit,  
les mini-batches de 32 (vs 256) stabilisent la descente de gradient avec peu d'images.

### Modèle linéaire

Architecture : $\vec{o} = A\vec{x} + b$ avec $A \in \mathbb{R}^{2 \times 16384}$, $b \in \mathbb{R}^{2}$.  
Nombre de paramètres : $2 \times 16384 + 2 = 32\,770$.

In [ ]:
print("=" * 60)
print(f"Modèle linéaire - CBIS-DDSM | Paramètres : {2*N_PIXELS + 2:,}")
print("=" * 60)

A_lin, b_lin, loss_lin, etr_lin, ete_lin = train_lin(
    X_train_flat, Y_train_oh, y_train,
    X_test_flat,  y_test,
    n_input=N_PIXELS, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=50
)

print(f"\nRésultat final :")
print(f"  Train : {etr_lin[-1]*100:.2f}%  |  Test : {ete_lin[-1]*100:.2f}%  |  Écart : {abs(etr_lin[-1]-ete_lin[-1])*100:.2f}%")

### Modèle H=1

Architecture : une couche cachée de 128 neurones avec ReLU.  
Nombre de paramètres : $128 \times 16384 + 128 + 2 \times 128 + 2 = 2\,098\,818$.

In [ ]:
print("=" * 60)
print(f"Modèle H=1 - CBIS-DDSM (p1=128) | Paramètres : {128*N_PIXELS+128+2*128+2:,}")
print("=" * 60)

A1_h1, b1_h1, A2_h1, b2_h1, loss_h1, etr_h1, ete_h1 = train_h1(
    X_train_flat, Y_train_oh, y_train,
    X_test_flat,  y_test,
    n_input=N_PIXELS, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=50
)

print(f"\nRésultat final :")
print(f"  Train : {etr_h1[-1]*100:.2f}%  |  Test : {ete_h1[-1]*100:.2f}%  |  Écart : {abs(etr_h1[-1]-ete_h1[-1])*100:.2f}%")

### Modèle H=2

Architecture : deux couches cachées (128 puis 64 neurones) avec ReLU.  
Nombre de paramètres : $128 \times 16384 + 128 + 64 \times 128 + 64 + 2 \times 64 + 2 = 2\,107\,330$.

In [ ]:
print("=" * 60)
print(f"Modèle H=2 - CBIS-DDSM (p1=128, p2=64) | Paramètres : {128*N_PIXELS+128+64*128+64+2*64+2:,}")
print("=" * 60)

A1_h2, b1_h2, A2_h2, b2_h2, A3_h2, b3_h2, \
    loss_h2, etr_h2, ete_h2 = train_h2(
    X_train_flat, Y_train_oh, y_train,
    X_test_flat,  y_test,
    n_input=N_PIXELS, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=50
)

print(f"\nRésultat final :")
print(f"  Train : {etr_h2[-1]*100:.2f}%  |  Test : {ete_h2[-1]*100:.2f}%  |  Écart : {abs(etr_h2[-1]-ete_h2[-1])*100:.2f}%")

In [ ]:
# Courbes d'erreur train/test pour les 3 architectures denses
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titres  = ["Linéaire", "H=1 (128)", "H=2 (128, 64)"]
donnees = [(etr_lin, ete_lin), (etr_h1, ete_h1), (etr_h2, ete_h2)]

for ax, titre, (etr, ete) in zip(axes, titres, donnees):
    ax.plot([e*100 for e in etr], label="Train", color="steelblue")
    ax.plot([e*100 for e in ete], label="Test",  color="coral")
    ax.set_title(titre)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Erreur (%)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("CBIS-DDSM - Architectures denses (bénin vs malin)", fontsize=13)
plt.tight_layout()
plt.savefig("courbes_cbis_dense.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Section 3.3 - CNN adapté au diagnostic médical (Option B : PyTorch)

On adapte l'architecture CNN de la Partie 2 à ce problème binaire sur images en niveaux de gris 128×128.

**Différences par rapport au CNN CIFAR-10 :**

| | CNN CIFAR-10 | CNN CBIS-DDSM |
|---|---|---|
| Entrée | (N, **3**, 32, 32) RGB | (N, **1**, 128, 128) niveaux de gris |
| Sortie | 10 classes | **2** classes (bénin/malin) |
| Max-poolings | 2 | **3** (car image plus grande) |
| Flatten | 8×8×64 = 4096 | 16×16×64 = **16 384** |
| Régularisation | aucune | **Dropout(0.5)** (peu de données) |
| Fonction de coût | CrossEntropyLoss | CrossEntropyLoss(**pondérée**) |

Le Dropout (p=0.5) éteint aléatoirement 50% des neurones à chaque passe avant pendant l'entraînement.  
Cela force le réseau à ne pas mémoriser les données, particulièrement important avec peu d'images médicales.

La `CrossEntropyLoss` pondérée pénalise davantage les erreurs sur les cas malins (minoritaires),  
ce qui améliore la sensibilité (rappel sur la classe maligne), métrique prioritaire en oncologie.

In [ ]:
class CNN_CBIS(nn.Module):
    """
    CNN pour la classification binaire de mammographies CBIS-DDSM.
    Entrée  : (batch, 1, 128, 128) image en niveaux de gris normalisée entre 0 et 1
    Sortie  : (batch, 2) logits bruts (CrossEntropyLoss intègre le softmax)

    Architecture :
    - Bloc 1 : Conv(1→32) + Conv(32→32) + MaxPool → (32, 64, 64)
    - Bloc 2 : Conv(32→64) + Conv(64→64) + MaxPool → (64, 32, 32)
    - Bloc 3 : Conv(64→64)               + MaxPool → (64, 16, 16)
    - Flatten : 64×16×16 = 16 384 valeurs
    - Dropout(0.5) + Linear(16384 → 2)
    """
    def __init__(self):
        super(CNN_CBIS, self).__init__()

        # Bloc 1 : 1 canal gris en entrée, 32 cartes de caractéristiques
        self.conv1 = nn.Conv2d(in_channels=1,  out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)   # 128x128 → 64x64

        # Bloc 2 : 32 → 64 cartes, capture des motifs plus abstraits
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)   # 64x64 → 32x32

        # Bloc 3 : 64 cartes supplémentaires pour les textures fines des mammographies
        self.conv5 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)   # 32x32 → 16x16

        # Tête de classification : 16x16x64 = 16384 → 2 classes
        self.dropout = nn.Dropout(p=0.5)       # régularisation contre l'overfitting
        self.fc      = nn.Linear(16 * 16 * 64, 2)
        self.relu    = nn.ReLU()

    def forward(self, x):
        """
        Passage avant : chemin des données à travers les couches.
        x entrée : (batch, 1, 128, 128)
        """
        x = self.relu(self.conv1(x))   # (batch, 32, 128, 128)
        x = self.relu(self.conv2(x))   # (batch, 32, 128, 128)
        x = self.pool1(x)              # (batch, 32, 64, 64)

        x = self.relu(self.conv3(x))   # (batch, 64, 64, 64)
        x = self.relu(self.conv4(x))   # (batch, 64, 64, 64)
        x = self.pool2(x)              # (batch, 64, 32, 32)

        x = self.relu(self.conv5(x))   # (batch, 64, 32, 32)
        x = self.pool3(x)              # (batch, 64, 16, 16)

        x = torch.flatten(x, 1)        # (batch, 16384)
        x = self.dropout(x)            # éteint 50% des neurones pendant l'entraînement
        x = self.fc(x)                 # (batch, 2) logits bruts
        return x


# Vérification de l'architecture
model_cnn  = CNN_CBIS()
n_params   = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(model_cnn)
print(f"\nTotal paramètres entraînables : {n_params:,}")

# Test avec un batch fictif (1 canal gris, 128x128)
x_fictif = torch.randn(4, 1, 128, 128)
sortie   = model_cnn(x_fictif)
print(f"Test forward : entrée {list(x_fictif.shape)} → sortie {list(sortie.shape)}")

### Préparation des données pour PyTorch

PyTorch attend des tenseurs au format $(N, C, H, W)$ (canaux en deuxième dimension).  
Ici $C=1$ (niveaux de gris), on ajoute donc une dimension de canal avec `np.newaxis`.

On normalise par la moyenne et l'écart-type calculés sur le train  
(on applique ces mêmes statistiques au test pour éviter toute fuite d'information).

In [ ]:
# Ajout dimension canal : (n, 128, 128) → (n, 1, 128, 128)
X_tr_t = torch.tensor(X_raw_train[:, np.newaxis, :, :], dtype=torch.float32)
X_te_t = torch.tensor(X_raw_test[:,  np.newaxis, :, :], dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.long)
y_te_t = torch.tensor(y_test,  dtype=torch.long)

# Normalisation par la moyenne et l'écart-type du train
mean_cbis = X_tr_t.mean()
std_cbis  = X_tr_t.std()
X_tr_t    = (X_tr_t - mean_cbis) / (std_cbis + 1e-8)
X_te_t    = (X_te_t - mean_cbis) / (std_cbis + 1e-8)   # stats du train appliquées au test

print(f"X_tr_t : {list(X_tr_t.shape)}")
print(f"X_te_t : {list(X_te_t.shape)}")
print(f"Normalisation : mean={mean_cbis:.4f}  std={std_cbis:.4f}")

# DataLoader : petits batches (32) car dataset médical limité
train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32,  shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=64,  shuffle=False)

print(f"\nBatches entraînement : {len(train_loader)}  |  Batches test : {len(test_loader)}")

### Entraînement du CNN médical

On suit, en plus du taux d'erreur global, la **sensibilité** (rappel sur la classe maligne) à chaque epoch :

$$\text{Sensibilité} = \frac{VP}{VP + FN}$$

où $VP$ = vrais positifs (malins correctement détectés) et $FN$ = faux négatifs (cancers manqués).  
C'est la métrique prioritaire : un $FN$ signifie un cancer non détecté.

In [ ]:
def train_cnn_medical(model, train_loader, test_loader, class_weights,
                      n_epochs=30, lr=1e-3):
    """
    Entraînement du CNN avec CrossEntropyLoss pondérée.
    Suit le taux d'erreur global ET la sensibilité (rappel malin) à chaque epoch.

    model         : instance de CNN_CBIS
    train_loader  : DataLoader entraînement
    test_loader   : DataLoader test
    class_weights : Tensor(2,) poids inversement proportionnels aux fréquences
    n_epochs      : nombre d'epochs
    lr            : learning rate
    """
    # CrossEntropyLoss pondérée : les erreurs sur les malins sont plus pénalisées
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    hist_loss_tr, hist_loss_te = [], []
    hist_err_tr,  hist_err_te  = [], []
    hist_sensib                = []   # sensibilité sur la classe maligne

    for epoch in range(n_epochs):

        # Phase d'entraînement
        model.train()                          # active le mode entraînement (dropout actif)
        loss_cumul, erreurs_tr = 0, 0

        for Xb, yb in train_loader:
            optimizer.zero_grad()              # réinitialise les gradients accumulés
            sortie = model(Xb)                 # passe avant : (batch, 2) logits
            loss   = criterion(sortie, yb)     # calcul de la loss pondérée
            loss.backward()                    # rétropropagation : gradients de tous les filtres
            optimizer.step()                   # mise à jour des poids
            loss_cumul  += loss.item() * len(Xb)
            erreurs_tr  += int((sortie.argmax(dim=1) != yb).sum())

        loss_tr = loss_cumul / len(train_loader.dataset)
        err_tr  = erreurs_tr / len(train_loader.dataset)

        # Phase d'évaluation
        model.eval()                           # désactive le dropout pour l'évaluation
        loss_cumul, erreurs_te = 0, 0
        vp_malin, fn_malin     = 0, 0          # pour la sensibilité

        with torch.no_grad():                  # pas de calcul de gradient en évaluation
            for Xb, yb in test_loader:
                sortie      = model(Xb)
                preds       = sortie.argmax(dim=1)
                loss_cumul += criterion(sortie, yb).item() * len(Xb)
                erreurs_te += int((preds != yb).sum())

                # Sensibilité : sur les cas réellement malins (yb == 1)
                masque_malin = (yb == 1)
                vp_malin    += int((preds[masque_malin] == 1).sum())
                fn_malin    += int((preds[masque_malin] == 0).sum())

        loss_te = loss_cumul / len(test_loader.dataset)
        err_te  = erreurs_te / len(test_loader.dataset)
        sensib  = vp_malin / (vp_malin + fn_malin + 1e-8)

        hist_loss_tr.append(loss_tr)
        hist_loss_te.append(loss_te)
        hist_err_tr.append(err_tr)
        hist_err_te.append(err_te)
        hist_sensib.append(sensib)

        print(f"  Epoch {epoch+1:2d}/{n_epochs} | "
              f"Loss tr={loss_tr:.4f} te={loss_te:.4f} | "
              f"Erreur tr={err_tr*100:.2f}% te={err_te*100:.2f}% | "
              f"Sensibilité={sensib*100:.1f}%")

    return hist_loss_tr, hist_loss_te, hist_err_tr, hist_err_te, hist_sensib


print("train_cnn_medical chargée.")

In [ ]:
torch.manual_seed(42)
model_cnn = CNN_CBIS()

print("=" * 65)
print(f"CNN CBIS-DDSM | Paramètres : {sum(p.numel() for p in model_cnn.parameters()):,}")
print("Note : sur CPU environ 10-20 min pour 30 epochs.")
print("=" * 65)

hist_loss_tr, hist_loss_te, hist_err_tr, hist_err_te, hist_sensib = train_cnn_medical(
    model_cnn, train_loader, test_loader,
    class_weights=class_weights_torch,
    n_epochs=30, lr=1e-3
)

print(f"\nRésultat final :")
print(f"  Train : {hist_err_tr[-1]*100:.2f}%")
print(f"  Test  : {hist_err_te[-1]*100:.2f}%")
print(f"  Écart : {abs(hist_err_tr[-1]-hist_err_te[-1])*100:.2f}%")
print(f"  Sensibilité finale (rappel malin) : {hist_sensib[-1]*100:.1f}%")

In [ ]:
# Courbes d'apprentissage du CNN médical
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

ax1.plot(hist_loss_tr, label="Train", color="steelblue")
ax1.plot(hist_loss_te, label="Test",  color="coral")
ax1.set_title("Loss (cross-entropy pondérée) - CNN")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot([e*100 for e in hist_err_tr], label="Train", color="steelblue")
ax2.plot([e*100 for e in hist_err_te], label="Test",  color="coral")
ax2.set_title("Taux d'erreur (%) - CNN")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Erreur (%)")
ax2.legend()
ax2.grid(alpha=0.3)

ax3.plot([s*100 for s in hist_sensib], color="darkgreen", label="Sensibilité (malin)")
ax3.axhline(y=80, color="orange", linestyle="--", alpha=0.8, label="Seuil 80%")
ax3.set_title("Sensibilité sur la classe maligne")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Sensibilité (%)")
ax3.legend()
ax3.grid(alpha=0.3)

plt.suptitle("CNN CBIS-DDSM - Mammographies bénin vs malin", fontsize=13)
plt.tight_layout()
plt.savefig("courbes_cnn_cbis.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Section 3.4 - Analyse de la matrice de confusion et discussion médicale

En diagnostic médical, le taux d'erreur global est une métrique insuffisante.  
Il faut distinguer deux types d'erreurs aux conséquences très différentes :

| Erreur | Prédiction | Réalité | Conséquence médicale |
|--------|-----------|---------|----------------------|
| **Faux négatif (FN)** | Bénin (0) | Malin (1) | **Cancer manqué → pronostic grave** |
| **Faux positif (FP)** | Malin (1) | Bénin (0) | Fausse alarme → biopsie inutile, stress |

On calcule quatre métriques standard en screening :
$$\text{Sensibilité} = \frac{VP}{VP + FN} \qquad \text{Spécificité} = \frac{VN}{VN + FP}$$
$$\text{VPP} = \frac{VP}{VP + FP} \qquad \text{VPN} = \frac{VN}{VN + FN}$$

In [ ]:
# Collecte de toutes les prédictions du CNN sur le test
model_cnn.eval()
all_preds   = []
all_probas  = []

with torch.no_grad():
    for Xb, yb in test_loader:
        sortie = model_cnn(Xb)
        probas = torch.softmax(sortie, dim=1)     # conversion en probabilités
        preds  = probas.argmax(dim=1)
        all_preds.append(preds.numpy())
        all_probas.append(probas[:, 1].numpy())   # probabilité d'être malin

y_pred         = np.concatenate(all_preds)         # (n_test,) prédictions binaires
y_proba_malin  = np.concatenate(all_probas)        # (n_test,) probabilité d'être malin

print(f"Prédictions collectées : {len(y_pred)} images")
print(f"  Prédits bénins (0) : {int((y_pred==0).sum())}")
print(f"  Prédits malins (1) : {int((y_pred==1).sum())}")

In [ ]:
# Calcul de la matrice de confusion et des métriques médicales
cm = confusion_matrix(y_test, y_pred)

VN = cm[0, 0]   # vrais négatifs  : bénin prédit bénin
FP = cm[0, 1]   # faux positifs   : bénin prédit malin
FN = cm[1, 0]   # faux négatifs   : malin prédit bénin  ← cancer manqué
VP = cm[1, 1]   # vrais positifs  : malin prédit malin

sensibilite = VP / (VP + FN + 1e-8)   # rappel sur la classe maligne
specificite = VN / (VN + FP + 1e-8)   # rappel sur la classe bénigne
vpp         = VP / (VP + FP + 1e-8)   # valeur prédictive positive
vpn         = VN / (VN + FN + 1e-8)   # valeur prédictive négative
erreur_glob = (FP + FN) / len(y_test)

print("Matrice de confusion :")
print(f"                   Prédit Bénin   Prédit Malin")
print(f"  Réel Bénin  (0) :      {VN:4d}          {FP:4d}")
print(f"  Réel Malin  (1) :      {FN:4d}          {VP:4d}")
print()
print(f"Métriques médicales :")
print(f"  Sensibilité (rappel malin)  : {sensibilite*100:.2f}%  ← VP / (VP + FN)")
print(f"  Spécificité (rappel bénin)  : {specificite*100:.2f}%  ← VN / (VN + FP)")
print(f"  VPP (précision sur malins)  : {vpp*100:.2f}%")
print(f"  VPN (précision sur bénins)  : {vpn*100:.2f}%")
print(f"  Erreur globale              : {erreur_glob*100:.2f}%")
print()
print(f"Faux négatifs (cancers manqués) : {FN} sur {VP+FN} cas malins réels")

In [ ]:
# Visualisation de la matrice de confusion + métriques
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Matrice de confusion ---
ax = axes[0]
im = ax.imshow(cm, cmap="Blues", interpolation="nearest")
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Prédit Bénin", "Prédit Malin"], fontsize=10)
ax.set_yticklabels(["Réel Bénin",   "Réel Malin"],   fontsize=10)
ax.set_xlabel("Prédiction", fontsize=11)
ax.set_ylabel("Réalité",    fontsize=11)
ax.set_title("Matrice de confusion - CNN CBIS-DDSM", fontsize=12)

# Annotations dans chaque case
for i, j, etiquette, couleur in [
    (0, 0, f"VN\n{VN}",  "#000000"),
    (0, 1, f"FP\n{FP}",  "#c0392b"),
    (1, 0, f"FN\n{FN}",  "#c0392b"),
    (1, 1, f"VP\n{VP}",  "#000000"),
]:
    ax.text(j, i, etiquette, ha="center", va="center",
            fontsize=13, fontweight="bold", color=couleur)

# --- Barres des métriques ---
ax2     = axes[1]
noms    = ["Sensibilité\n(rappel malin)", "Spécificité\n(rappel bénin)", "VPP", "VPN"]
valeurs = [sensibilite*100, specificite*100, vpp*100, vpn*100]
coul    = ["#c0392b", "#2980b9", "#8e44ad", "#27ae60"]

barres = ax2.bar(noms, valeurs, color=coul, edgecolor="white", width=0.5)
ax2.set_ylim(0, 110)
ax2.axhline(y=80, color="orange", linestyle="--", alpha=0.8, label="Seuil 80%")
ax2.set_ylabel("Valeur (%)", fontsize=11)
ax2.set_title("Métriques médicales - CNN CBIS-DDSM", fontsize=12)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

for barre, val in zip(barres, valeurs):
    ax2.text(barre.get_x() + barre.get_width()/2, val + 1,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("matrice_confusion_cbis.png", dpi=100, bbox_inches="tight")
plt.show()

### Visualisation des erreurs du CNN

On affiche les faux négatifs (cancers manqués) et faux positifs (fausses alarmes)  
avec la probabilité d'être malin prédite par le modèle pour chacune.

In [ ]:
idx_fn = np.where((y_test == 1) & (y_pred == 0))[0]   # cancers manqués
idx_fp = np.where((y_test == 0) & (y_pred == 1))[0]   # fausses alarmes

n_fn_affich = min(5, len(idx_fn))
n_fp_affich = min(5, len(idx_fp))

print(f"Faux négatifs (cancers manqués) : {len(idx_fn)}")
print(f"Faux positifs (fausses alarmes) : {len(idx_fp)}")

if n_fn_affich > 0 or n_fp_affich > 0:
    n_cols = max(n_fn_affich, n_fp_affich, 1)
    fig, axes = plt.subplots(2, n_cols, figsize=(3*n_cols, 6))
    if n_cols == 1:
        axes = axes.reshape(2, 1)   # assure que axes est toujours 2D

    for j in range(n_cols):
        # Ligne 0 : faux négatifs
        if j < n_fn_affich:
            idx = idx_fn[j]
            axes[0][j].imshow(X_raw_test[idx], cmap="gray", vmin=0, vmax=1)
            axes[0][j].set_title(
                f"FN - Cancer manqué\nProba malin : {y_proba_malin[idx]*100:.1f}%",
                fontsize=9, color="darkred")
        axes[0][j].axis("off")

        # Ligne 1 : faux positifs
        if j < n_fp_affich:
            idx = idx_fp[j]
            axes[1][j].imshow(X_raw_test[idx], cmap="gray", vmin=0, vmax=1)
            axes[1][j].set_title(
                f"FP - Fausse alarme\nProba malin : {y_proba_malin[idx]*100:.1f}%",
                fontsize=9, color="darkorange")
        axes[1][j].axis("off")

    plt.suptitle("Erreurs du CNN CBIS-DDSM", fontsize=13)
    plt.tight_layout()
    plt.savefig("erreurs_cnn_cbis.png", dpi=100, bbox_inches="tight")
    plt.show()

### Tableau comparatif final

In [ ]:
# Calcul de la sensibilité pour les modèles denses (passe avant NumPy)
def sensibilite_dense(X_flat, y, model_type, **params):
    """
    Calcule la sensibilité (rappel malin) pour un modèle dense NumPy.
    Effectue la passe avant complète selon le type de modèle.

    X_flat     : (n, N_PIXELS)
    y          : (n,) étiquettes binaires
    model_type : 'lin', 'h1' ou 'h2'
    params     : A, b (lin) ou A1, b1, A2, b2 (h1) ou ... (h2)
    Retourne   : float sensibilité
    """
    if model_type == "lin":
        o = X_flat @ params["A"].T + params["b"]
    elif model_type == "h1":
        z1 = np.maximum(0, X_flat @ params["A1"].T + params["b1"])
        o  = z1 @ params["A2"].T + params["b2"]
    elif model_type == "h2":
        z1 = np.maximum(0, X_flat @ params["A1"].T + params["b1"])
        z2 = np.maximum(0, z1 @ params["A2"].T + params["b2"])
        o  = z2 @ params["A3"].T + params["b3"]

    preds        = np.argmax(o, axis=1)
    masque_malin = (y == 1)
    vp = int((preds[masque_malin] == 1).sum())
    fn = int((preds[masque_malin] == 0).sum())
    return vp / (vp + fn + 1e-8)


s_lin = sensibilite_dense(X_test_flat, y_test, "lin", A=A_lin, b=b_lin)
s_h1  = sensibilite_dense(X_test_flat, y_test, "h1",  A1=A1_h1, b1=b1_h1, A2=A2_h1, b2=b2_h1)
s_h2  = sensibilite_dense(X_test_flat, y_test, "h2",  A1=A1_h2, b1=b1_h2,
                           A2=A2_h2, b2=b2_h2, A3=A3_h2, b3=b3_h2)

print("=" * 75)
print(f"{'Modèle':<25} {'Params':>12} {'Train':>8} {'Test':>8} {'Sensib.':>10}")
print("=" * 75)

configs = [
    ("Linéaire",      2*N_PIXELS+2,                        etr_lin[-1], ete_lin[-1], s_lin),
    ("H=1 (128)",     128*N_PIXELS+128+2*128+2,             etr_h1[-1],  ete_h1[-1],  s_h1),
    ("H=2 (128,64)",  128*N_PIXELS+128+64*128+64+2*64+2,   etr_h2[-1],  ete_h2[-1],  s_h2),
    ("CNN (PyTorch)", n_params,                             hist_err_tr[-1], hist_err_te[-1], sensibilite),
]

for modele, params, e_tr, e_te, s in configs:
    print(f"{modele:<25} {params:>12,} {e_tr*100:>7.2f}% {e_te*100:>7.2f}% {s*100:>9.1f}%")

print("=" * 75)
print("\nNote : la sensibilité (rappel malin) est la métrique prioritaire en diagnostic.")
print("Un faux négatif = cancer non détecté = retard de diagnostic potentiellement fatal.")

### Discussion finale

**Déséquilibre des classes et choix de la métrique prioritaire.**  
Le dataset CBIS-DDSM présente un déséquilibre entre cas bénins et malins. Un modèle naïf qui prédirait systématiquement "bénin" obtiendrait un faible taux d'erreur global mais une sensibilité nulle, ce qui serait médicalement catastrophique. La pondération de la `CrossEntropyLoss` oblige le modèle à pénaliser davantage les erreurs sur les malins, ce qui améliore la sensibilité au prix d'une légère dégradation de la spécificité. Ce compromis est cliniquement justifié : en dépistage, on tolère davantage les fausses alarmes (biopsie inutile) que les cancers manqués.

**Importance des faux négatifs en diagnostic médical.**  
Un faux négatif signifie qu'un cancer a été classifié bénin. Pour la patiente, cela peut entraîner un retard de diagnostic de plusieurs mois, pendant lesquels la tumeur peut progresser vers un stade moins traitable. À l'inverse, un faux positif génère du stress et une biopsie inutile, mais ne met pas la vie en danger. En radiologie clinique, la sensibilité est généralement fixée à 90–95% dans les guidelines de dépistage.

**Limites de nos architectures sur des données médicales.**  
Trois limites principales sont identifiées.

Premièrement, le redimensionnement de l'image originale à 128×128 entraîne une perte d'information. Les microcalcifications, premiers signes d'un carcinome in situ, sont des structures de quelques pixels sur l'image originale et peuvent disparaître après redimensionnement agressif. Une résolution de 224×224 serait préférable si la mémoire le permet.

Deuxièmement, le faible nombre d'images médicales disponibles (quelques centaines après déduplication) par rapport à CIFAR-10 (50 000) favorise fortement l'overfitting. Le Dropout(0.5) ajouté atténue ce problème mais ne le résout pas. La data augmentation (retournements horizontaux, rotations légères ±10°, variations de contraste) permettrait de multiplier artificiellement le nombre d'exemples en restant médicalement plausible.

Troisièmement, nous travaillons sur la mammographie complète alors que le signal clinique pertinent est localisé dans une région d'intérêt (ROI). Le CSV fournit `cropped image file path` qui pointe vers la ROI déjà découpée par les radiologues. Travailler directement sur ces crops serait plus pertinent cliniquement : on classerait la lésion elle-même plutôt que l'image entière où la lésion ne représente qu'une fraction des pixels.

**Comparaison des architectures.**  
Les modèles denses peinent sur ce dataset pour les mêmes raisons qu'avec CIFAR-10 : ils traitent les 16 384 pixels comme des variables indépendantes sans notion de structure spatiale. Les spiculations, les contours irréguliers ou les densités locales caractéristiques des masses malignes sont des patterns locaux que seule la convolution peut capturer. Le CNN améliore significativement la sensibilité grâce à ses filtres qui détectent ces motifs quelle que soit leur position dans la mammographie.

**Perspectives d'amélioration.**  
Le transfer learning (fine-tuning d'un modèle pré-entraîné sur ImageNet comme ResNet50 ou EfficientNet-B0) permettrait de contourner le problème du faible nombre de données : le modèle a déjà appris des milliers de détecteurs de textures et de formes, il suffit d'adapter les dernières couches à la mammographie. C'est l'approche standard en imagerie médicale computationnelle, capable d'atteindre des AUC supérieures à 0.85 sur CBIS-DDSM avec quelques centaines d'images.